In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
import math
from scipy.stats import norm
import matplotlib.pyplot as plt
import sys
from openpyxl import Workbook

In [2]:
#Entry
tperiod = 8

In [3]:

# Location data
df_V = pd.read_excel('fullDist Set V Data Location-smallsize.xlsx')
df_V.head
"""
df_V = pd.read_excel('fullDist Set V Data Location.xlsx')
df_V.head
"""

"\ndf_V = pd.read_excel('fullDist Set V Data Location.xlsx')\ndf_V.head\n"

In [4]:
# County data

df_C = pd.read_excel('fullDist Set C Data-smallsize.xlsx')
df_C.head
"""
df_C = pd.read_excel('fullDist Set C Data.xlsx')
df_C.head
"""

"\ndf_C = pd.read_excel('fullDist Set C Data.xlsx')\ndf_C.head\n"

In [5]:
# Resource Data

df_R = pd.read_excel('fullDist R-smallsize.xlsx')
df_R.head
"""
df_R = pd.read_excel('fullDist R.xlsx')
df_R.head
"""

"\ndf_R = pd.read_excel('fullDist R.xlsx')\ndf_R.head\n"

In [6]:
#Time and distance matrices 

timeMatrix = np.loadtxt("durationFULL-smallsize.txt"); # load the time duration matrix (seconds)
distanceMatrix = np.loadtxt("distanceFULL-smallsize.txt");
"""
timeMatrix = np.loadtxt("durationFULL.txt"); # load the time duration matrix (seconds)
distanceMatrix = np.loadtxt("distanceFULL.txt");
"""

'\ntimeMatrix = np.loadtxt("durationFULL.txt"); # load the time duration matrix (seconds)\ndistanceMatrix = np.loadtxt("distanceFULL.txt");\n'

In [7]:

df_inputs = pd.read_excel('fullDistInputs-smallsize.xlsx')
df_inputs.head
"""
df_inputs = pd.read_excel('fullDistInputs.xlsx')
df_inputs.head
"""

"\ndf_inputs = pd.read_excel('fullDistInputs.xlsx')\ndf_inputs.head\n"

In [8]:
# Load the K_ijt data from the Excel file

df_K = pd.read_excel('K_ijt_data_fixed-smallsize.xlsx')
df_K.head()
"""
df_K = pd.read_excel('K_ijt_data_fixed.xlsx')
df_K.head()
"""

"\ndf_K = pd.read_excel('K_ijt_data_fixed.xlsx')\ndf_K.head()\n"

In [9]:
"""
# Load the demand data from the Excel file
file_path = 'demand_data_smallsize.xlsx'  # Update with your actual file name

d_its = {}  # Initialize the demand dictionary

# Read the demand data from the single sheet
df = pd.read_excel(file_path, header=0, index_col=0)  # Set the first column as index

# Fill NaN values with 0
df = df.fillna(0)

# Assume we have one scenario with index s = 0
s = 0
demand_array = df.values

# Populate the dictionary
for i in range(demand_array.shape[0]):  # Iterate over locations (rows)
    for t in range(demand_array.shape[1]):  # Iterate over time periods (columns)
        d_its[(i, t, s)] = demand_array[i, t]

# Print a sample to verify the data
print("Sample demand data for location 6, time 1, scenario 0:", d_its.get((6, 1, s), "Not available"))
"""

'\n# Load the demand data from the Excel file\nfile_path = \'demand_data_smallsize.xlsx\'  # Update with your actual file name\n\nd_its = {}  # Initialize the demand dictionary\n\n# Read the demand data from the single sheet\ndf = pd.read_excel(file_path, header=0, index_col=0)  # Set the first column as index\n\n# Fill NaN values with 0\ndf = df.fillna(0)\n\n# Assume we have one scenario with index s = 0\ns = 0\ndemand_array = df.values\n\n# Populate the dictionary\nfor i in range(demand_array.shape[0]):  # Iterate over locations (rows)\n    for t in range(demand_array.shape[1]):  # Iterate over time periods (columns)\n        d_its[(i, t, s)] = demand_array[i, t]\n\n# Print a sample to verify the data\nprint("Sample demand data for location 6, time 1, scenario 0:", d_its.get((6, 1, s), "Not available"))\n'

In [10]:

# Estimated demand
file_path = 'demand_data_smallsize_more sce.xlsx'
sheet_names = ['Demand_1', 'Demand_2','Demand_3','Demand_4','Demand_5','Demand_6','Demand_7','Demand_8','Demand_9','Demand_10']

d_its = {}

for s, sheet in enumerate(sheet_names):  # Scenario index starts from 0
    df = pd.read_excel(file_path, sheet_name=sheet, header=0, index_col=0)
    
    # Fill NaN with 0 if needed
    df = df.fillna(0)
    
    demand_array = df.values

    for i in range(demand_array.shape[0]):
        actual_location = i  # Start location index from 1 instead of 0
        for t in range(demand_array.shape[1]):
            d_its[(actual_location, t, s)] = demand_array[i, t]

# Print a sample to verify the correct location
print("Sample demand data for location 6, time 1, scenario 4:", d_its[(6, 1, 4)])
print("Sample demand data for location 6, time 1, scenario 2:", d_its[(0, 1, 1)])


Sample demand data for location 6, time 1, scenario 4: 97.30811409820313
Sample demand data for location 6, time 1, scenario 2: 0.0


In [11]:
# Defining sets

# County Set
C = len(df_C)
print('All counties:', C)

# Location Sets
V = len(df_V)
print('All locations:', V)

# Subsets of Location Set (V0, V1, V2, V3)
V0 = []  # Warehouses and Suppliers
V1 = []  # Shelters
V2 = []  # RSAs (Staging Areas)
V3 = []  # PoDs (Points of Distribution)


for i in range(V):
    if df_V['Type'][i] == 'Warehouse' or df_V['Type'][i] == 'Supplier':
        V0.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'Shelter':
        V1.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'RSA':
        V2.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'PoD':
        V3.append(df_V['Index'][i])

# Print lengths of subsets
print('Warehouse/Supplier locations:', len(V0), V0)
print('Shelter locations:', len(V1), V1)
print('RSA locations:', len(V2), V2)
print('PoD locations:', len(V3), V3)

# Commodity Set
R = len(df_R)
print('All resources:', R)

# Time Period Set
tperiod = 8
T = int(df_inputs['Hours in 6 days'][0] / tperiod)
print('Time periods:', T)

#Time Period Subsets
T_Bef = list(range(int(T / 2)))
T_Aft = list(range(int(T / 2), T))
print('Time Before Landfall:', T_Bef)
print('Time After Landfall:', T_Aft)
"""
# Define scenario indices for 50 scenarios
sce = list(range(1))  # This will create a list [0, 1, 2, ..., 49]

# Define equal probabilities for each of the 50 scenarios
P = [1/1] * 1  # This will create a list with 50 elements, each with a value of 1/50

"""
# Define scenario indices for 50 scenarios
sce = list(range(10))  # This will create a list [0, 1, 2, ..., 49]

# Define equal probabilities for each of the 50 scenarios
P = [1/10] * 10  # This will create a list with 50 elements, each with a value of 1/50

# Print to verify
print("Scenario indices:", sce)
print("Scenario probabilities:", P)

All counties: 2
All locations: 7
Warehouse/Supplier locations: 1 [0]
Shelter locations: 2 [1, 2]
RSA locations: 2 [3, 4]
PoD locations: 2 [5, 6]
All resources: 1
Time periods: 3
Time Before Landfall: [0]
Time After Landfall: [1, 2]
Scenario indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Scenario probabilities: [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]


In [12]:
#Create Arc Set where cannot go from shelter to PoD

A = set() #set to store arc pairs

no_demand=[]
for i in set(V1).union(set(V3)):
    i_has_demand = any(df_V['County'][i] == df_C['County'][c] and df_C['Shelter Total Demand'][c] > 0 for c in range(C))
    if not i_has_demand:
        no_demand.append(i)
print(no_demand)

for i in range(V):
    for j in range(V):
        if i != j and i not in no_demand and j not in no_demand:
            if (i not in V1 or j not in V3) and (i not in V3 or j not in V1):
                A.add((i, j))

#print("Arc set:", arc_set)
print("Number of arcs:", len(A))
print("Number of arcs eliminated by this arc set:", (V*V)-len(A))
#print(A)

[1, 5]
Number of arcs: 18
Number of arcs eliminated by this arc set: 31


In [13]:
#Defining parameters

W_k = df_inputs['People on Truck'][0] #number of people on a truck
tp_salary = df_inputs['People on Truck'][0]*tperiod; # salary per time period 
fuel = df_inputs['Fuel Cost Per Mile'][0]*0.000621371; # fuel cost per meter
open_array = [df_inputs['Open Cost V0'][0],df_inputs['Open Cost V1'][0],df_inputs['Open Cost V2'][0],df_inputs['Open Cost V3'][0]] #Assumed opening costs

#Shelter Capacity at location i when i in V1 q_i[I]
q_i=[]
for i in V1:
    q_i.append(df_V['Pallet Capacity'][i])

Inv = []  # Initialize inventory list

for i in range(V):
    Inv.append(df_V['Water StartInv'][i])
I_ir0 = [[inv] for inv in Inv]
print (I_ir0)

"""
#starting inventory I_ir0[I][R]
Inv = []
for i in range(V):
    Inv.extend([df_V['S-Kit StartInv'][i],
                df_V['P-Kit StartInv'][i],
                df_V['Water StartInv'][i],
                df_V['Food StartInv'][i],
                df_V['Cot StartInv'][i],
                df_V['Blanket StartInv'][i]])
I_ir0 = [Inv[i:i+R] for i in range(0, len(Inv), R)]
"""

#penalty cost for location i p_i[I]
p_i=[]
for i in range(V):
    p_i.append(df_V['P_i'][i])

#time periods to travel i,j t_ijt[i][j]
t_ijt=[]
for i in range(V):
    row=[]
    for j in range(V):
        t_ijt_hours=timeMatrix[i][j]/3600
        t_ijt_period=math.ceil(t_ijt_hours/tperiod)
        row.append(t_ijt_period)
    t_ijt.append(row)
#print (t_ijt)

#distance to travel i,j e_ij[i][j]
e_ij=[]
for i in range(V):
    row=[]
    for j in range(V):
        e_ij_miles=distanceMatrix[i][j]*0.000621371
        row.append(e_ij_miles)
    e_ij.append(row)

#truck delay h_i[i]
h_i=[]
for i in range(V):
    h_i.append(df_V['TruckDelay'][i])

#Fraction of a truck 1 pallet of resource r requires w_r[R]
w_r=[]
for r in range(R):
    w_r.append(1/df_R['Pallet Units on Truck'][r])

#Cost to procure resource r from location i cost_ir[i][r]
cost=[]
for i in V0:
    cost.extend([0,0,df_V['Water PC'][i],df_V['Food PC'][i],0,0])
    cost_ir = [cost[i:i+R] for i in range(0, len(cost), R)]
    
#Factor of pallet required per commodity r of kit g_shelter[R]
g_shelter=[]
for r in range(R):
    g_shelter.append(df_R['pallet per S-Kit'][r])

#Factor of pallet required per commodity r of kit g_pod[R]
g_pod=[]
for r in range(R):
    g_pod.append(df_R['pallet per P-Kit'][r])

# Initialize K_ijt as a nested dictionary {i: {j: {t: capacity}}}
K_ijt = {}

# Populate K_ijt with data from df_K
for index, row in df_K.iterrows():
    i = row['From_Node']
    j = row['To_Node']
    t = row['Time_Period']
    capacity = row['Capacity']
    
    if i not in K_ijt:
        K_ijt[i] = {}
    if j not in K_ijt[i]:
        K_ijt[i][j] = {}
    
    K_ijt[i][j][t] = capacity

# Check the first few entries in K_ijt to ensure it’s structured correctly
#print (Inv)
#print("g_shelter:", g_shelter)
#print("g_pod:", g_pod)
#print(w_r)

[[390], [0], [0], [0], [0], [0], [0]]


# Defining Master Problem

In [14]:
# Create the master problem model
master_model = gp.Model("MasterProblem")

# Define the inventory decision variables I[i, r, t] for first-stage decisions
# These represent the inventory of commodity r at location i at the start of time period t
I = {}
for i in range(V):
    for r in range(R):
        for t in range(T):
            I[i, r, t] = master_model.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"I_{i}_{r}_{t}")

# Define the auxiliary variable θ with an initial lower bound, used for Benders cuts
theta = master_model.addVar(vtype=GRB.CONTINUOUS, name="theta")

# Set an initial lower bound for theta to prevent unboundedness from below
L = 0  # Choose a reasonable initial lower bound
master_model.addConstr(theta >= L, name="theta_lower_bound")

for i in range(V):
    for r in range(R):
        master_model.addConstr(I[i, r, 0] == I_ir0[i][r], name=f"initial_inventory_{i}_{r}")
"""
# Add constraints to ensure inventory at warehouses only decreases over time
for i in V0:
    for r in range(R):
        for t in range(1, T):
            master_model.addConstr(
                I[i, r, t] <= I[i, r, t-1],
                name=f"inventory_decrease_warehouse_{i}_{r}_{t}")

# Add constraints to ensure inventory at PoDs only increases before landfall
for i in V3:
    for r in range(R):
        for t in T_Bef[1:]:
            master_model.addConstr(
                I[i, r, t] >= I[i, r, t-1],
                name=f"inventory_increase_pod_{i}_{r}_{t}")
"""
# Set the objective function to minimize θ (representing the expected second-stage cost)
master_model.setObjective(theta, GRB.MINIMIZE)

# Set output flag to 0 to avoid printing Gurobi logs
master_model.setParam("OutputFlag", 0);

# Update the model to integrate new variables
master_model.update()
#master_model.write("master_problem.lp")

Set parameter Username
Academic license - for non-commercial use only - expires 2025-11-30


# Defining Optimality Subproblem

In [15]:
# Define the optimality subproblem
sub_opt = gp.Model("subprob_optimality")
sub_opt.modelSense = GRB.MINIMIZE

# Define second-stage decision variables U, S, and f for the optimality subproblem
U_opt = {}
S_opt = {}
f_opt = {}

# U: Demand shortage at location i at time period t in scenario s
for i in range(V):
    for t in range(T):
        U_opt[i, t] = sub_opt.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"U_opt_{i}_{t}")

# S: Demand satisfied at location i at time period t in scenario s
for i in range(V):
    for t in range(T):
        S_opt[i, t] = sub_opt.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"S_opt_{i}_{t}")

# f: Quantity of commodity r traveling on link (i, j) at time period t in scenario s
for i in range(V):
    for j in range(V):
        if (i, j) in A:
            for r in range(R):
                for t in range(T):
                    f_opt[i, j, r, t] = sub_opt.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"f_opt_{i}_{j}_{r}_{t}")

# Objective: Minimize the expected demand shortage cost
sub_opt.setObjective(gp.quicksum(p_i[i] * U_opt[i, t] for i in range(V) for t in range(T)))

# Constraint for inventory flow balance with U and S
pi_opt_inventory_balance_U = {}
for i in range(V):
    for t in range(1, T):  # Start from t=1 since t=0 is the initial condition
        pi_opt_inventory_balance_U[i, t] = sub_opt.addConstr(U_opt[i, t] - U_opt[i, t-1] + S_opt[i, t] == 0, name=f"inventory_balance_U_opt_{i}_{t}")

# Initial shortage for demand at t=0
pi_opt_initial_shortage_U = {}
for i in range(V):
    pi_opt_initial_shortage_U[i] = sub_opt.addConstr(U_opt[i, 0] == 0, name=f"initial_shortage_U_opt_{i}")

# Demand satisfaction constraint
pi_opt_demand_satisfaction = {}
for i in range(V):
    for t in range(T):
        pi_opt_demand_satisfaction[i, t] = sub_opt.addConstr(U_opt[i, t] - S_opt[i, t] >= 0, name=f"demand_satisfaction_opt_{i}_{t}")


# Flow conservation constraint for f with the RHS set to 0 temporarily, using appropriate g_shelter or g_pod
pi_opt_flow_conservation = {}
for i in range(V):
    for r in range(R):
        for t in range(T - 1):
            # Add the constraint to the model
            pi_opt_flow_conservation[i, r, t] = sub_opt.addConstr(
                -gp.quicksum(f_opt[i, j, r, t] for j in range(V) if (i, j) in A)
                + gp.quicksum(f_opt[j, i, r, t - t_ijt[j][i] + 1] for j in range(V) if (i, j) in A and t - t_ijt[j][i] + 1 >= 0) #
                - S_opt[i, t] * (g_shelter[r] if i in V1 else g_pod[r] if i in V3 else 0) == 0,
                name=f"flow_conservation_opt_{i}_{r}_{t}")

# Inventory balance constraint with RHS set to 0 for now
pi_opt_inventory_balance = {}
for i in range(V):
    for r in range(R):
        for t in range(T):       
            # Add the constraint to the model
            pi_opt_inventory_balance[i, r, t] = sub_opt.addConstr(
                -gp.quicksum(f_opt[i, j, r, t] for j in range(V) if (i, j) in A) #
                - S_opt[i, t] * (g_shelter[r] if i in V1 else g_pod[r] if i in V3 else 0) >= 0,
                name=f"inventory_balance_opt_{i}_{r}_{t}")


# Arc capacity constraint
pi_opt_arc_capacity = {}
for i in range(V):
    for j in range(V):
        if (i, j) in A:  # Only consider valid links in A
            for t in range(T):
                # Check if the capacity for this (i, j, t) exists in K_ijt
                #if i in K_ijt and j in K_ijt[i] and t in K_ijt[i][j]:
                    pi_opt_arc_capacity[i, j, t] = sub_opt.addConstr(-gp.quicksum(w_r[r] * f_opt[i, j, r, t] for r in range(R)) >= -K_ijt[i][j][t],
                        name=f"arc_capacity_opt_{i}_{j}_{t}")

sub_opt.setParam("OutputFlag", 0);

# Defining Feasibility Subproblem

In [16]:
# Define the feasibility subproblem
sub_feas = gp.Model("subprob_feasibility")
sub_feas.modelSense = GRB.MINIMIZE

# Define second-stage decision variables U, S, and f for the feasibility subproblem
U_feas = {}
S_feas = {}
f_feas = {}

# U: Demand shortage at location i at time period t in scenario s
for i in range(V):
    for t in range(T):
        U_feas[i, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, obj=0, name=f"U_feas_{i}_{t}")

# S: Demand satisfied at location i at time period t in scenario s
for i in range(V):
    for t in range(T):
        S_feas[i, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, obj=0, name=f"S_feas_{i}_{t}")

# f: Quantity of commodity r traveling on link (i, j) at time period t in scenario s
for i in range(V):
    for j in range(V):
        if (i, j) in A:
            for r in range(R):
                for t in range(T):
                    f_feas[i, j, r, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, obj=0, name=f"f_feas_{i}_{j}_{r}_{t}")


# Now define slack variables ε (epsilon) for each group in the objective function
# ε variables with objective coefficients corresponding to their respective constraint group
epsilon_1 = {}
for i in range(V):
    for t in range(1, T):
        epsilon_1[i, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"epsilon1_{i}_{t}")

epsilon_2 = {}
for i in range(V):
    for t in range(1, T):
        epsilon_2[i, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"epsilon2_{i}_{t}")

epsilon_3 = {}
for i in range(V):
    for t in range(T):
        epsilon_3[i, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"epsilon3_{i}_{t}")

epsilon_4 = {}
for i in range(V):
    for r in range(R):
        for t in range(T-1):
            epsilon_4[i, r, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"epsilon4_{i}_{r}_{t}")

epsilon_5 = {}
for i in range(V):
    for r in range(R):
        for t in range(T-1):
            epsilon_5[i, r, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"epsilon5_{i}_{r}_{t}")

epsilon_6 = {}
for i in range(V):
    for r in range(R):
        for t in range(T):
            epsilon_6[i, r, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"epsilon6_{i}_{r}_{t}")

epsilon_7 = {}
for i in range(V):
    for j in range(V):
        if (i, j) in A:
            for t in range(T):
                epsilon_7[i, j, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"epsilon7_{i}_{j}_{t}")

sub_feas.setObjective(
    gp.quicksum(epsilon_1[i, t] + epsilon_2[i, t] for i in range(V) for t in range(1, T)) +
    gp.quicksum(epsilon_3[i, t] for i in range(V) for t in range(T)) +
    gp.quicksum(epsilon_4[i, r, t] + epsilon_5[i, r, t] for i in range(V) for r in range(R) for t in range(T - 1)) +
    gp.quicksum(epsilon_6[i, r, t] for i in range(V) for r in range(R) for t in range(T)) +
    gp.quicksum(epsilon_7[i, j, t] for i in range(V) for j in range(V) if (i, j) in A for t in range(T)))

# Constraint: Inventory flow balance for U, S, and epsilon variables
pi_feas_inventory_balance_U = {}
for i in range(V):
    for t in range(1, T):  # Note: starting from t=1 as per the constraint
        pi_feas_inventory_balance_U[i, t] = sub_feas.addConstr(
            U_feas[i, t] - U_feas[i, t-1] + S_feas[i, t] + epsilon_1[i, t] - epsilon_2[i, t] == 0,
            name=f"inventory_balance_U_feas_{i}_{t}"
        )

# Initialize shortage for U at t=0
pi_feas_initial_shortage_U = {}
for i in range(V):
    pi_feas_initial_shortage_U[i] = sub_feas.addConstr(
        U_feas[i, 0] == 0,
        name=f"initial_shortage_U_feas_{i}"
    )


# Demand satisfaction constraint
pi_feas_demand_satisfaction = {}
for i in range(V):
    for t in range(T):
        pi_feas_demand_satisfaction[i, t] = sub_feas.addConstr(
            U_feas[i, t] - S_feas[i, t] + epsilon_3[i, t] >= 0,
            name=f"demand_satisfaction_feas_{i}_{t}"
        )

# Flow conservation constraint
pi_feas_flow_conservation = {}
for i in range(V):
    for r in range(R):
        for t in range(T - 1):
            pi_feas_flow_conservation[i, r, t] = sub_feas.addConstr(
                -gp.quicksum(f_feas[i, j, r, t] for j in range(V) if (i, j) in A)
                + gp.quicksum(f_feas[j, i, r, t - t_ijt[j][i] + 1] for j in range(V) if (i, j) in A and t - t_ijt[j][i] + 1 >= 0) #
                - S_feas[i, t] * (g_shelter[r] if i in V1 else g_pod[r] if i in V3 else 0)
                + epsilon_4[i, r, t] - epsilon_5[i, r, t] == 0,
                name=f"flow_conservation_feas_{i}_{r}_{t}"
            )

# Constraint for flow and demand satisfaction with epsilon_6
pi_feas_inventory_balance = {}
for i in range(V):
    for r in range(R):
        for t in range(T):
            pi_feas_inventory_balance[i, r, t] = sub_feas.addConstr(
                -gp.quicksum(f_feas[i, j, r, t] for j in range(V) if (i, j) in A)
                - S_feas[i, t] * (g_shelter[r] if i in V1 else g_pod[r] if i in V3 else 0)
                + epsilon_6[i, r, t] >= 0,  # Right-hand side set to 0 for now
                name=f"inventory_balance_feas_{i}_{r}_{t}"
            )

# Flow capacity constraint for all links, time periods, and scenarios
pi_feas_arc_capacity = {}
for i in range(V):
    for j in range(V):
        if (i, j) in A:  # Only consider valid links in A
            for t in range(T):
                # Check if K_ijt[i][j][t] exists before adding the constraint
                #if i in K_ijt and j in K_ijt[i] and t in K_ijt[i][j]:
                    pi_feas_arc_capacity[i, j, t] = sub_feas.addConstr(
                        -gp.quicksum(w_r[r] * f_feas[i, j, r, t] for r in range(R))
                        + epsilon_7[i, j, t] >= -K_ijt[i][j][t],
                        name=f"arc_capacity_feas_{i}_{j}_{t}"
                    )

sub_feas.setParam("OutputFlag", 0);

# Creating Benders Loop

In [17]:
LB = -1e10
UB = 1e10

Ocuts = 0
Fcuts = 0

iter = 0
eps = 1e-5  # tolerance parameter

while (UB - LB) / max(1e-10,abs(UB)) > eps:
    iter += 1
    #if iter > 5:
     #   break
    
    # First, solve the master problem to get the values of first-stage variables and theta
    I_values = {}  # Store I values for each (i, r, t)
    theta_value = 0  # Initialize theta value

    # Optimize the master problem
    master_model.optimize()

    # Check if the optimization was successful
    if master_model.status == GRB.OPTIMAL:
        # Extract the values of first-stage decision variables I[i, r, t]
        for i in range(V):
            for r in range(R):
                for t in range(T):
                    I_values[i, r, t] = I[i, r, t].x

        # Get the value of theta
        theta_value = theta.x
        """
        # Extract the value of theta
        print(f"Iteration {iter}:")
        print(f"  Objective value (Lower Bound): {master_model.objVal}")
        print(f"  Theta value: {theta.x}")
    
        # Print the values of first-stage decision variables I[i, r, t]
        print("  First-stage decision variables (I[i, r, t]):")
        for i in range(V):
            for r in range(R):
                for t in range(T):
                    print(f"    I[{i}, {r}, {t}] = {I[i, r, t].x}")
        """
        # Update the lower bound
        LB = master_model.objVal       
         
    elif master_model.status == GRB.INFEASIBLE:
        print("Master problem is infeasible.")
        exit(0)  # Or handle infeasibility as required for your problem

    elif master_model.status == GRB.UNBOUNDED:
        print("Master problem is unbounded.")
        exit(0)  # Or handle unboundedness as required for your problem

    else:
        print(f"Optimization was stopped with status {master_model.status}")
        exit(0)
    
    # Number of scenarios
    num = len(sce)

    # Optimality subproblem dual multipliers
    pi_opt_inventory_balance_U_val = {s: {} for s in range(num)}
    pi_opt_initial_shortage_U_val = {s: {} for s in range(num)}
    pi_opt_demand_satisfaction_val = {s: {} for s in range(num)}
    pi_opt_flow_conservation_val = {s: {} for s in range(num)}
    pi_opt_inventory_balance_val = {s: {} for s in range(num)}
    pi_opt_arc_capacity_val = {s: {} for s in range(num)}
    
    # Feasibility subproblem dual multipliers
    pi_feas_inventory_balance_U_val = {}
    pi_feas_initial_shortage_U_val = {}
    pi_feas_demand_satisfaction_val = {}
    pi_feas_flow_conservation_val = {}
    pi_feas_inventory_balance_val = {}
    pi_feas_arc_capacity_val = {}

    # Initialize a vector to store the optimal objective value for the optimality subproblem for each scenario
    Q_opt = np.zeros(num)
    # Initialize a vector to store the optimal objective value for the feasibility subproblem for each scenario
    w_feas = np.zeros(num)
    FLAG = 0; #(0 if I_hat is feasible ∀ s, and 1 otherwise)

    # Update RHS for optimality subproblem constraints for every scenario
    infeasible_sce = -1
    for s in range(num):  # Loop over all scenarios
        # Update RHS of inventory flow balance constraints
        for i in range(V):
            for t in range(1, T):
                pi_opt_inventory_balance_U[i, t].setAttr(GRB.Attr.RHS, d_its[(i, t, s)])
        
        # Update RHS of initial shortage constraints
        for i in range(V):
            pi_opt_initial_shortage_U[i].setAttr(GRB.Attr.RHS, d_its[(i, 0, s)])
        
        # Update RHS of flow conservation constraints
        for i in range(V):
            for r in range(R):
                for t in range(T - 1):
                    pi_opt_flow_conservation[i, r, t].setAttr(GRB.Attr.RHS, I_values[i, r, t+1] - I_values[i, r, t])
        
        # Update RHS of inventory balance constraints
        for i in range(V):
            for r in range(R):
                for t in range(T):
                    pi_opt_inventory_balance[i, r, t].setAttr(GRB.Attr.RHS, -I_values[i, r, t])
        

        sub_opt.optimize()
    
        if sub_opt.status == GRB.OPTIMAL:
            # Update the optimal objective value for each scenario
            Q_opt[s] = sub_opt.objVal
    
            # Retrieve and store dual values for each relevant constraint
            for key, constraint in pi_opt_inventory_balance_U.items():
                pi_opt_inventory_balance_U_val[s][key] = constraint.pi
        
            for key, constraint in pi_opt_initial_shortage_U.items():
                pi_opt_initial_shortage_U_val[s][key] = constraint.pi
        
            for key, constraint in pi_opt_demand_satisfaction.items():
                pi_opt_demand_satisfaction_val[s][key] = constraint.pi
        
            for key, constraint in pi_opt_flow_conservation.items():
                pi_opt_flow_conservation_val[s][key] = constraint.pi
        
            for key, constraint in pi_opt_inventory_balance.items():
                pi_opt_inventory_balance_val[s][key] = constraint.pi
        
            for key, constraint in pi_opt_arc_capacity.items():
                pi_opt_arc_capacity_val[s][key] = constraint.pi
        
        elif sub_opt.status == GRB.INFEASIBLE:
            # Set FLAG to 1 if infeasible, indicating the need for a feasibility cut
            FLAG = 1
            infeasible_sce = s
            break;
        else:
            # Handle other statuses
            print('One of the scenario subproblems is infeasible')
            print('status =', sub_opt.status)
            exit(0)
    print("FLAG =", FLAG)
    if FLAG == 1:
        
            # Update RHS for feasibility subproblem constraints for every scenario
            s = infeasible_sce
            print("s=", s)
            
            # Update RHS of inventory flow balance constraints for U
            for i in range(V):
                for t in range(1, T):
                    pi_feas_inventory_balance_U[i, t].setAttr(GRB.Attr.RHS, d_its[(i, t, s)])
            
            # Update RHS of initial shortage constraints for U at t=0
            for i in range(V):
                pi_feas_initial_shortage_U[i].setAttr(GRB.Attr.RHS, d_its[(i, 0, s)])
            
            
            # Update RHS of flow conservation constraints
            for i in range(V):
                for r in range(R):
                    for t in range(T - 1):
                        pi_feas_flow_conservation[i, r, t].setAttr(GRB.Attr.RHS, I_values[i, r, t+1] - I_values[i, r, t])
            
            # Update RHS of inventory balance constraints
            for i in range(V):
                for r in range(R):
                    for t in range(T):
                        pi_feas_inventory_balance[i, r, t].setAttr(GRB.Attr.RHS, -I_values[i, r, t])
                                

            # Optimize the feasibility subproblem
            sub_feas.optimize()
            w_feas[s] = sub_feas.objVal  # Store the optimal objective value for this scenario
            #print("sub_feas.objval = ", sub_feas.objVal)

            # Retrieve and store dual values for each relevant constraint in the feasibility subproblem
            for key, constraint in pi_feas_inventory_balance_U.items():
                pi_feas_inventory_balance_U_val[key] = constraint.pi
            for key, constraint in pi_feas_initial_shortage_U.items():
                pi_feas_initial_shortage_U_val[key] = constraint.pi
            for key, constraint in pi_feas_demand_satisfaction.items():
                pi_feas_demand_satisfaction_val[key] = constraint.pi            
            for key, constraint in pi_feas_flow_conservation.items():
                pi_feas_flow_conservation_val[key] = constraint.pi            
            for key, constraint in pi_feas_inventory_balance.items():
                pi_feas_inventory_balance_val[key] = constraint.pi            
            for key, constraint in pi_feas_arc_capacity.items():
                pi_feas_arc_capacity_val[key] = constraint.pi   
            
            if w_feas[s] > eps:
                # Initialize the feasibility cut expression
                feasibility_cut_expr = gp.LinExpr()
            
                # Add the terms for inventory flow balance for U and S constraints
                for key, dual_val in pi_feas_inventory_balance_U_val.items():
                    i, t = key
                    feasibility_cut_expr += dual_val * d_its[(i, t, s)]
                    
                # Add the terms for initial shortage for U constraints
                for key, dual_val in pi_feas_initial_shortage_U_val.items():
                    i = key
                    feasibility_cut_expr += dual_val * d_its[(i, 0, s)]
                
                # Add the terms for demand satisfaction constraints
                for key, dual_val in pi_feas_demand_satisfaction_val.items():
                    i, t = key
                    feasibility_cut_expr += dual_val * 0

                # Add the terms for flow conservation constraints
                for key, dual_val in pi_feas_flow_conservation_val.items():
                    i, r, t = key
                    feasibility_cut_expr += dual_val * (I[i, r, t+1] - I[i, r, t])
    
                # Add the terms for inventory balance constraints
                for key, dual_val in pi_feas_inventory_balance_val.items():
                    i, r, t = key
                    feasibility_cut_expr += dual_val * (-I[i, r, t])

                # Add the terms for arc capacity constraints
                for key, dual_val in pi_feas_arc_capacity_val.items():
                    i, j, t = key
                    feasibility_cut_expr += dual_val * -K_ijt[i][j][t]
                """
                #Evaluate the LHS of the feasibility cut
                lhs_value = feasibility_cut_expr.getValue()
                
                #Compare LHS with the feasibility problem's objective value
                print(f"Iteration {iter}, Scenario {s}")
                print(f"Feasibility Problem Objective Value: {w_feas[s]}")
                print(f"LHS of Feasibility Cut: {lhs_value}")
                print(f"Difference: {abs(lhs_value - w_feas[s])}")
                """
                #print("feasibility cut:", feasibility_cut_expr)
                # Add the feasibility cut to the master problem
                master_model.addConstr(feasibility_cut_expr <= 0, name=f"feasibility_cut_scenario_{s}")
                Fcuts += 1  # Increment the counter for feasibility cuts
                #master_model.write(f"master_problem_with_cuts_iter_{iter}.lp")
                

    else:
        print("here")
        print("Q_opt = ", Q_opt)
        print("w_feas =", w_feas)
        # Calculate the temporary upper bound using only the second-stage costs
        tempUB = sum(Q_opt[s] * P[s] for s in range(num))        
        if tempUB < UB:
            UB = tempUB
        if (UB - LB) / max(1e-10, abs(LB)) < eps:
            break

        else:
            # Initialize the optimality cut expression
            optimality_cut_expr = gp.LinExpr()
            for s in range(num):
                 
                # Add the terms for inventory flow balance with U and S constraints
                for key, dual_val in pi_opt_inventory_balance_U_val[s].items():
                    i, t = key
                    optimality_cut_expr += P[s] * dual_val * d_its[(i, t, s)]
                
                # Add the terms for initial shortage for U constraints
                for key, dual_val in pi_opt_initial_shortage_U_val[s].items():
                    i = key
                    optimality_cut_expr += P[s] * dual_val * d_its[(i, 0, s)]
                
                # Add the terms for demand satisfaction constraints
                for key, dual_val in pi_opt_demand_satisfaction_val[s].items():
                    i, t = key
                    optimality_cut_expr += P[s] * dual_val * 0
                
                # Add the terms for flow conservation constraints
                for key, dual_val in pi_opt_flow_conservation_val[s].items():
                    i, r, t = key
                    optimality_cut_expr += P[s] * dual_val * (I[i, r, t + 1] - I[i, r, t])
                
                # Add the terms for inventory balance constraints
                for key, dual_val in pi_opt_inventory_balance_val[s].items():
                    i, r, t = key
                    optimality_cut_expr += P[s] * dual_val * (-I[i, r, t])
                
                # Add the terms for arc capacity constraints
                for key, dual_val in pi_opt_arc_capacity_val[s].items():
                    i, j, t = key
                    optimality_cut_expr += P[s] * dual_val * -K_ijt[i][j][t]

                #print("optimality cut expression:", optimality_cut_expr)
            #print("optimality cut:", optimality_cut_expr)
            
            # Add the optimality cut to the master model
            master_model.addConstr(theta >= optimality_cut_expr, name="optimality_cut")
            Ocuts += 1  # Increment the counter for optimality cuts

            

    # Print information at each iteration
    print("iter =", iter)
    print("LB =", LB)
    print("UB =", UB)
    print("===============================================================")


# Final summary after convergence
print("#########################################################################################")
print("#########################################################################################")
print("Final iteration =", iter)
print("Number of feasibility cuts added =", Fcuts)
print("Number of optimality cuts added =", Ocuts)
print("Lower Bound (LB) =", LB)
print("Upper Bound (UB) =", UB)
print("FLAG =", FLAG)


# Print first-stage decision variable values (I_values)
#print("I_values (First-stage decision variables):")
#for key, value in I_values.items():
    #print(f"I[{key}] =", value)

FLAG = 1
s= 0
iter = 1
LB = 0.0
UB = 10000000000.0
FLAG = 1
s= 0
iter = 2
LB = 0.0
UB = 10000000000.0
FLAG = 1
s= 0
iter = 3
LB = 0.0
UB = 10000000000.0
FLAG = 1
s= 0
iter = 4
LB = 0.0
UB = 10000000000.0
FLAG = 0
here
Q_opt =  [ 996.02726564 1095.6299922   896.42453908 1195.23271877  796.82181251
 1294.83544533  697.21908595 1145.43135549  846.62317579 1245.03408205]
w_feas = [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
iter = 5
LB = 0.0
UB = 1020.9279472813729
FLAG = 1
s= 0
iter = 6
LB = 0.0
UB = 1020.9279472813729
FLAG = 1
s= 0
iter = 7
LB = 0.0
UB = 1020.9279472813729
FLAG = 0
here
Q_opt =  [ 915.95757334 1007.55333067  824.361816   1099.149088    732.76605867
 1190.74484534  641.17030134 1053.35120934  778.56393734 1144.94696667]
w_feas = [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
iter = 8
LB = 0.0
UB = 938.856512669528
FLAG = 1
s= 0
iter = 9
LB = 0.0
UB = 938.856512669528
FLAG = 1
s= 0
iter = 10
LB = 0.0
UB = 938.856512669528
FLAG = 1
s= 0
iter = 11
LB = 0.0
UB = 938.856512669528
FLAG = 1
s= 0
iter = 12
